# Movies Recommender System

In this project, we build movie recommenders on Kaggle's [The Movies Dataset](https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset): TMDB metadata for ~45k movies plus a 100k-rating MovieLens sample.

Files expected in `DATA_DIR`: `movies_metadata.csv`, `credits.csv`, `keywords.csv`, `ratings_small.csv`, `links_small.csv`.

We build:
1. A non-personalised **popularity chart** (IMDB weighted rating)
2. Three **content-based** recommenders: plot text with TF-IDF, cast/director/genres/keywords tokens, and **semantic similarity** with sentence embeddings
3. **Collaborative filtering** (item-item cosine similarity on mean-centred ratings)

The recommenders are inspected qualitatively on a few well-known movies. A quantitative offline evaluation is left for future work.

In [ ]:
from pathlib import Path
from ast import literal_eval
import re

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import linear_kernel
from sklearn.preprocessing import normalize
from scipy import sparse
from nltk.stem.snowball import SnowballStemmer

DATA_DIR = Path('.')

In [ ]:
md = pd.read_csv(DATA_DIR / 'movies_metadata.csv', low_memory=False)
md.shape

In [ ]:
# Check for missing values
md.isnull().sum()

#### Cleaning
* A few rows have malformed ids (dates in the `id` column) and some movies appear twice with the same id: drop both.
* Titles are **not** unique (remakes, re-releases), so we keep every movie and resolve title lookups to the most-voted version later.

In [ ]:
md['id'] = pd.to_numeric(md['id'], errors='coerce')
md = md.dropna(subset=['id'])
md['id'] = md['id'].astype(int)
md = md.drop_duplicates(subset='id', keep='first').reset_index(drop=True)

md['genres'] = md['genres'].fillna('[]').apply(literal_eval).apply(lambda gs: [g['name'] for g in gs])
md['release_date'] = pd.to_datetime(md['release_date'], errors='coerce')
md['year'] = md['release_date'].dt.year.astype('Int64')
md['vote_count'] = pd.to_numeric(md['vote_count'], errors='coerce')
md['vote_average'] = pd.to_numeric(md['vote_average'], errors='coerce')
md.shape

## Popularity chart

We use IMDB's *weighted rating* formula to construct the chart:

Weighted Rating (WR) = $(\frac{v}{v + m} \cdot R) + (\frac{m}{v + m} \cdot C)$

where,
* *v* is the number of votes for the movie
* *m* is the minimum votes required to be listed in the chart
* *R* is the average rating of the movie
* *C* is the mean vote across the whole pool

It is a Bayesian average: movies with few votes are shrunk towards the mean *C*, so a 10/10 from 3 votes cannot top the chart. We use the **95th percentile** of vote counts as *m*. Both *m* and *C* are computed over movies with at least one vote (unrated movies have `vote_average = 0`, which would drag *C* down), and within the genre pool when building genre charts.

In [ ]:
def weighted_rating(v, R, m, C):
    return (v / (v + m)) * R + (m / (v + m)) * C


def build_chart(df, genres=None, percentile=0.95, top_n=250):
    # Top-N movies by weighted rating; m and C are computed within the (genre-filtered) pool
    if genres:
        df = df[df['genres'].apply(lambda gs: all(g in gs for g in genres))]
    rated = df[df['vote_count'] > 0]
    C = rated['vote_average'].mean()
    m = rated['vote_count'].quantile(percentile)
    qualified = rated[rated['vote_count'] >= m].copy()
    qualified['score'] = weighted_rating(qualified['vote_count'], qualified['vote_average'], m, C)
    print(f'C = {C:.3f}, m = {m:.0f}, qualified movies = {len(qualified):,}')
    cols = ['title', 'year', 'vote_count', 'vote_average', 'score']
    return qualified.sort_values('score', ascending=False)[cols].head(top_n)

#### Top 10 movies

In [ ]:
build_chart(md).head(10)

## Create chart based on genre(s)

In [ ]:
build_chart(md, ['Action', 'Romance']).head(10)

In [ ]:
build_chart(md, ['Romance']).head(10)

## Content Based Recommender

The chart gives the same recommendations to everyone. To personalise, we compute similarity between movies from their content and suggest movies similar to one the user liked (**Content Based Filtering**).

We build three content-based recommenders based on:
* Movie Overviews and Taglines (TF-IDF, matches exact words)
* Movie Cast, Director, Genres and Keywords (token overlap)
* Movie Overviews, Taglines, Genres and Keywords as sentence embeddings (**semantic similarity**, matches meaning)

All return the nearest neighbours by cosine similarity. Instead of materialising the full N x N similarity matrix (45k x 45k float64 is ~16 GB), we compute similarities for the query row only, which needs O(N) memory per query.

In [ ]:
# Titles are not unique: map each title to the row of its most-voted version
title_to_idx = (md.sort_values('vote_count', ascending=False)
                  .drop_duplicates('title')
                  .reset_index()
                  .set_index('title')['index'])


def recommend_movie(title, features, top_n=10):
    # features: one L2-normalised row per movie in md (sparse or dense), so a dot product is the cosine similarity
    if title not in title_to_idx.index:
        raise KeyError(f'Movie not found: {title!r}')
    idx = title_to_idx[title]
    sims = linear_kernel(features[idx:idx + 1], features).ravel()
    sims[idx] = -np.inf  # skip the movie itself
    top = np.argpartition(-sims, top_n)[:top_n]
    top = top[np.argsort(-sims[top])]
    return md.iloc[top][['title', 'year']].assign(similarity=sims[top].round(3))

##### 1. Recommendation using Movie Overviews and Taglines

In [ ]:
def preprocess(text):
    # A custom preprocessor replaces scikit-learn's default one, so lowercasing has to happen here
    # (lowercase=True is ignored when preprocessor is set). Numbers become a single placeholder token.
    return re.sub(r'\d+', ' num ', text.lower())


tfidf = TfidfVectorizer(min_df=2, max_df=0.85, stop_words='english', preprocessor=preprocess)

md['description'] = md['overview'].fillna('') + ' ' + md['tagline'].fillna('')
tfidf_matrix = tfidf.fit_transform(md['description'])  # rows are L2-normalised
tfidf_matrix.shape

In [ ]:
recommend_movie("The Godfather", tfidf_matrix)

In [ ]:
recommend_movie("The Dark Knight", tfidf_matrix)

In [ ]:
recommend_movie("The Avengers", tfidf_matrix)

The description recommender mostly finds movies from the same franchise (Batman films for **The Dark Knight**). Someone who liked **The Dark Knight** may like it because of Nolan, so next we use more suggestive metadata than **Overview** and **Tagline**.

##### 2. Metadata Based Recommender

In [ ]:
credits = pd.read_csv(DATA_DIR / 'credits.csv').drop_duplicates('id')
keywords = pd.read_csv(DATA_DIR / 'keywords.csv').drop_duplicates('id')

# Join on the TMDB id column. (DataFrame.join(on='id') would match md.id against the *row index*
# of credits and attach another movie's cast and crew.)
n_movies = len(md)
md = md.merge(credits, on='id', how='left').merge(keywords, on='id', how='left')
assert len(md) == n_movies, 'merge must not duplicate movies'

In [ ]:
for col in ['cast', 'crew', 'keywords']:
    md[col] = md[col].fillna('[]').apply(literal_eval)


def get_director(crew):
    return next((c['name'] for c in crew if c.get('job') == 'Director'), None)


md['director'] = md['crew'].apply(get_director)
md['cast'] = md['cast'].apply(lambda xs: [x['name'] for x in xs[:3]])
md['keywords'] = md['keywords'].apply(lambda xs: [x['name'] for x in xs])

In [ ]:
# Sanity check on a known movie
md.loc[title_to_idx['The Godfather'], ['title', 'director', 'cast']]

We create a metadata "soup" for every movie of **top-3 cast, director, genres and keywords**, vectorise it with a **Count Vectorizer** and compare movies by cosine similarity.

1. **Strip spaces and lowercase** every name, so that **Johnny Depp** and **Johnny Galecki** do not share the token *johnny*.
2. **Prefix each token with its field** (`cast:`, `director:`, `genre:`, `keyword:`) so values from different fields never collide.
3. **Mention the director 3 times** to give it more weight relative to the cast.
4. **Keywords** that appear only once in the dataset are dropped, and the rest are stemmed so *dogs* and *dog* match. (The original keywords stay in `md['keywords']`; the semantic model below reads them as text.)

In [ ]:
keyword_counts = md['keywords'].explode().value_counts()
frequent_keywords = set(keyword_counts[keyword_counts > 1].index)
stemmer = SnowballStemmer('english')
md['keywords_stemmed'] = md['keywords'].apply(lambda ks: [stemmer.stem(k) for k in ks if k in frequent_keywords])

DIRECTOR_WEIGHT = 3


def clean_token(name):
    return name.lower().replace(' ', '')


def metadata_tokens(row):
    tokens = [f'cast:{clean_token(c)}' for c in row['cast']]
    if isinstance(row['director'], str):
        tokens += [f'director:{clean_token(row["director"])}'] * DIRECTOR_WEIGHT
    tokens += [f'genre:{clean_token(g)}' for g in row['genres']]
    tokens += [f'keyword:{clean_token(k)}' for k in row['keywords_stemmed']]
    return tokens


def identity(tokens):
    return tokens


md['metadata_tokens'] = md.apply(metadata_tokens, axis=1)
count = CountVectorizer(analyzer=identity)  # tokens are already prepared
count_matrix = normalize(count.fit_transform(md['metadata_tokens']))
count_matrix.shape

In [ ]:
recommend_movie('The Dark Knight', count_matrix)

In [ ]:
recommend_movie('The Godfather', count_matrix)

##### 3. Semantic similarity with sentence embeddings

TF-IDF only matches **exact words**: a heist movie about "thieves who plan a robbery" and one about "a crew breaking into a vault" share almost no vocabulary, so they look unrelated. A pretrained **sentence-embedding model** maps a text to a dense vector so that texts with similar *meaning* land close together, even with different words. Cosine similarity between those vectors is the **semantic similarity**.

* Model: [`all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) (22M parameters, 384 dimensions, no training needed). It truncates inputs to 256 word pieces, which covers most overviews.
* Input text per movie: genres, keywords, overview and tagline. The title is left out so the model does not just match franchise names. Cast and director are left out as well, because a pretrained model knows little about most names; the metadata recommender handles those as exact tokens.
* Movies with no overview and no tagline get a zero vector, so they are never returned as neighbours.
* Embedding ~45k movies on a laptop CPU takes a few minutes, so the vectors are cached to `movie_embeddings.npz` and reused when the data and the model are unchanged.

In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = 'all-MiniLM-L6-v2'
EMBEDDINGS_PATH = DATA_DIR / 'movie_embeddings.npz'

has_text = (md['overview'].fillna('') + md['tagline'].fillna('')).str.strip() != ''
md['semantic_text'] = (
    'Genres: ' + md['genres'].str.join(', ')
    + '. Keywords: ' + md['keywords'].str.join(', ')
    + '. ' + md['overview'].fillna('') + ' ' + md['tagline'].fillna(''))


def embed_movies(texts, ids):
    if EMBEDDINGS_PATH.exists():
        cached = np.load(EMBEDDINGS_PATH)
        if str(cached['model']) == EMBEDDING_MODEL and np.array_equal(cached['ids'], ids):
            return cached['embeddings']
    model = SentenceTransformer(EMBEDDING_MODEL)
    embeddings = model.encode(texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
    np.savez(EMBEDDINGS_PATH, model=np.array(EMBEDDING_MODEL), ids=ids, embeddings=embeddings)
    return embeddings


embeddings = embed_movies(md['semantic_text'].tolist(), md['id'].to_numpy())
embeddings[~has_text.to_numpy()] = 0  # no text, nothing to compare
embeddings.shape

In [ ]:
recommend_movie("The Godfather", embeddings)

In [ ]:
recommend_movie("The Dark Knight", embeddings)

In [ ]:
recommend_movie("The Avengers", embeddings)

#### Side by side

The same movie under the three content-based recommenders. Things to look for: TF-IDF tends to return the same franchise, the metadata recommender follows people and genres, and the semantic recommender follows the *kind of story*. None of this is measured yet; it is a qualitative comparison.

In [ ]:
def compare(title, top_n=5):
    features = {'TF-IDF text': tfidf_matrix, 'Metadata': count_matrix, 'Semantic': embeddings}
    return pd.concat({name: recommend_movie(title, f, top_n)['title'].reset_index(drop=True)
                      for name, f in features.items()}, axis=1)

In [ ]:
compare("The Godfather")

In [ ]:
compare("The Dark Knight")

In [ ]:
compare("The Avengers")

## Collaborative Filtering

Content-based engines only look at what a movie *is*. **Collaborative Filtering** uses what other users did: if people who liked the movies I liked also liked X, X is a good suggestion for me. We build **item-item CF**:

* Ratings are **mean-centred per user**, so a generous rater's 4 and a harsh rater's 4 are not treated alike. Centred cosine is the same as Pearson correlation on co-rated movies.
* Two movies are similar if the same users rate them above or below their own average in the same way (cosine of the centred rating columns).
* A user's score for an unseen movie *i* is the sum over movies *j* they rated of `sim(i, j) x centred rating(u, j)`.
* Movies with fewer than 10 ratings are never recommended: with one or two co-raters, cosine similarity is mostly noise.

`ratings_small.csv` uses MovieLens `movieId`s; `links_small.csv` maps them to TMDB ids, i.e. to rows of `md`, so every recommender shares one set of movies.

In [ ]:
ratings = pd.read_csv(DATA_DIR / 'ratings_small.csv')
links = pd.read_csv(DATA_DIR / 'links_small.csv')

links = links.dropna(subset=['tmdbId']).astype({'tmdbId': int})
tmdb_to_row = pd.Series(md.index, index=md['id'])
links['row'] = links['tmdbId'].map(tmdb_to_row)
links = links.dropna(subset=['row']).astype({'row': int}).drop_duplicates('movieId')

n_raw = len(ratings)
ratings = (ratings.merge(links[['movieId', 'row']], on='movieId', how='inner')
                  .sort_values(['userId', 'timestamp'])
                  .drop_duplicates(['userId', 'row'], keep='last'))
print(f'{len(ratings):,} of {n_raw:,} ratings mapped to TMDB movies; '
      f'{ratings.userId.nunique()} users, {ratings.row.nunique():,} movies')

In [ ]:
MIN_RATINGS_PER_MOVIE = 10

users = np.sort(ratings['userId'].unique())
items = np.sort(ratings['row'].unique())  # movies that have ratings: rows of md
user_pos = pd.Series(np.arange(len(users)), index=users)
item_pos = pd.Series(np.arange(len(items)), index=items)


def to_matrix(values):
    return sparse.csr_matrix(
        (values, (user_pos[ratings['userId']].to_numpy(), item_pos[ratings['row']].to_numpy())),
        shape=(len(users), len(items)))


user_mean = ratings.groupby('userId')['rating'].mean()
R = to_matrix(ratings['rating'].to_numpy())  # users x movies, ratings are 0.5-5 so non-zero = rated
R_centered = to_matrix((ratings['rating'] - ratings['userId'].map(user_mean)).to_numpy())
Xn = normalize(R_centered, axis=0).tocsc()  # each movie's centred rating column has unit length

n_ratings = np.asarray((R > 0).sum(axis=0)).ravel()
eligible = n_ratings >= MIN_RATINGS_PER_MOVIE
print(f'rating matrix: {R.shape[0]} users x {R.shape[1]:,} movies, '
      f'{R.nnz / (R.shape[0] * R.shape[1]):.1%} filled; {eligible.sum():,} movies have >= {MIN_RATINGS_PER_MOVIE} ratings')

In [ ]:
def top_movies(scores, top_n):
    top = np.argpartition(-scores, top_n)[:top_n]
    top = top[np.argsort(-scores[top])]
    return top[scores[top] > 0]  # never pad the list with movies that have no positive evidence


def recommend_movie_cf(title, top_n=10):
    # Movies rated similarly to `title` by the same users (item-item CF)
    idx = title_to_idx.get(title)
    if idx is None or idx not in item_pos.index:
        raise KeyError(f'Movie not found among rated movies: {title!r}')
    j = item_pos[idx]
    sims = (Xn.T @ Xn[:, j]).toarray().ravel()
    sims[j] = -np.inf
    sims[~eligible] = -np.inf
    top = top_movies(sims, top_n)
    return md.iloc[items[top]][['title', 'year']].assign(similarity=sims[top].round(3))


def recommend_for_user(user_id, top_n=10):
    # Personalised: score(u, i) = sum_j sim(i, j) * centred rating(u, j), computed as Xn.T @ (Xn @ r_u)
    if user_id not in user_pos.index:
        raise KeyError(f'Unknown user: {user_id}')
    u = user_pos[user_id]
    scores = (Xn.T @ (Xn @ R_centered[u].T)).toarray().ravel()
    scores[R[u].indices] = -np.inf  # already rated
    scores[~eligible] = -np.inf
    top = top_movies(scores, top_n)
    return md.iloc[items[top]][['title', 'year']].assign(score=scores[top].round(3))


def liked_by_user(user_id, top_n=5):
    rated = ratings[ratings['userId'] == user_id].sort_values('rating', ascending=False).head(top_n)
    return md.iloc[rated['row'].to_numpy()][['title', 'year']].assign(rating=rated['rating'].to_numpy())

In [ ]:
recommend_movie_cf("The Godfather")

In [ ]:
recommend_movie_cf("The Dark Knight")

A user's highest-rated movies, and what item-item CF recommends to them:

In [ ]:
liked_by_user(1)

In [ ]:
recommend_for_user(1)

## Where this goes next

* **Quantitative evaluation.** So far the recommenders are only inspected by eye. A time-based train/test split per user with ranking metrics (Precision/Recall/NDCG@10) against popularity baselines would show which approach actually works.
* **Combining the signals.** Fuse the semantic, metadata and collaborative rankings (for example with reciprocal rank fusion), and eventually learn the combination with a ranking model such as LightGBM LambdaMART.
* **Matrix factorisation** (SVD / ALS) for personalisation that copes better with sparse ratings.